*micrograd is a tiny autograd engine and neural network library written in Python. It allows you to build and train simple neural networks from scratch, providing a clear understanding of how backpropagation works. The library is designed to be minimalistic and educational, making it a great resource for learning about deep learning concepts.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def f(x):
    return 3*x**2-4*x+5
x =-3
f(3)
xs =np.arange(-10,10,0.01)
xs
h=0.0001
(f(x+h)-f(x))/h

In [ ]:
import math
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self.grad = 0.0
        self._backward = lambda: None  

    def __repr__(self):
        return f"Value(data={self.data}), label={self.label}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1-t**2) * out.grad
        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other

    def __radd__(self, other):
        return self + other

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def __pow__(self, power):
        assert isinstance(power, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data ** power, (self,), f'**{power}')

        def _backward():
            self.grad += (power * self.data ** (power - 1)) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def backward(self):
        # Topological order all of the children in the graph
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited :
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()




In [ ]:
import numpy
import random
class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self,x):
        act = sum([wi*xi for wi,xi in zip(self.w,x)])+self.b
        return act.tanh()

    def params(self):
        return self.w+[self.b]

class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self,x):
        outs = [n(x) for n in self.neurons]
        return outs

    def params(self):
        return [p for nueron in self.neurons for p in nueron.params()] 

class MLP:
    def __init__(self,nin,nout):
        sz = [nin] + nout
        self.layers = [Layer(sz[i],sz[i+1]) for i in range(len(nout))]

    def __call__(self,x):
        for layer in self.layers:
            x=layer(x)
        return x

    def params(self):
        return [p for layer in self.layers for p in layer.params()]

x=[1.0,2.0]
mlp = MLP(2,[4,4,16])
mlp(x)


In [ ]:
# parse and visualize the logfile
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

sz = "124M"

loss_baseline = {
    "124M": 3.2924,
}[sz]
hella2_baseline = { # HellaSwag for GPT-2
    "124M": 0.294463,
    "350M": 0.375224,
    "774M": 0.431986,
    "1558M": 0.488946,
}[sz]
hella3_baseline = { # HellaSwag for GPT-3
    "124M": 0.337,
    "350M": 0.436,
    "774M": 0.510,
    "1558M": 0.547,
}[sz]

# load the log file
with open("log124M_40B/log.txt", "r") as f:
    lines = f.readlines()

# parse the individual lines, group by stream (train,val,hella)
streams = {}
for line in lines:
    step, stream, val = line.strip().split()
    if stream not in streams:
        streams[stream] = {}
    streams[stream][int(step)] = float(val)

# convert each stream from {step: val} to (steps[], vals[])
# so it's easier for plotting
streams_xy = {}
for k, v in streams.items():
    # get all (step, val) items, sort them
    xy = sorted(list(v.items()))
    # unpack the list of tuples to tuple of lists
    streams_xy[k] = list(zip(*xy))

# create figure
plt.figure(figsize=(16, 6))

# Panel 1: losses: both train and val
plt.subplot(121)
xs, ys = streams_xy["train"] # training loss
ys = np.array(ys)
plt.plot(xs, ys, label=f'nanogpt ({sz}) train loss')
print("Min Train Loss:", min(ys))
xs, ys = streams_xy["val"] # validation loss
plt.plot(xs, ys, label=f'nanogpt ({sz}) val loss')
# horizontal line at GPT-2 baseline
if loss_baseline is not None:
    plt.axhline(y=loss_baseline, color='r', linestyle='--', label=f"OpenAI GPT-2 ({sz}) checkpoint val loss")
plt.xlabel("steps")
plt.ylabel("loss")
plt.yscale('log')
plt.ylim(top=4.0)
plt.legend()
plt.title("Loss")
print("Min Validation Loss:", min(ys))

# Panel 2: HellaSwag eval
plt.subplot(122)
xs, ys = streams_xy["hella"] # HellaSwag eval
ys = np.array(ys)
plt.plot(xs, ys, label=f"nanogpt ({sz})")
# horizontal line at GPT-2 baseline
if hella2_baseline:
    plt.axhline(y=hella2_baseline, color='r', linestyle='--', label=f"OpenAI GPT-2 ({sz}) checkpoint")
if hella3_baseline:
    plt.axhline(y=hella3_baseline, color='g', linestyle='--', label=f"OpenAI GPT-3 ({sz}) checkpoint")
plt.xlabel("steps")
plt.ylabel("accuracy")
plt.legend()
plt.title("HellaSwag eval")
print("Max Hellaswag eval:", max(ys))


In [ ]:
import torch
import matplotlib.pyplot as plt

theta = 10000.0
dim =8
max_sqlen = 8
a =torch.arange(0,dim,2)
arr = 1.0/theta**(a[:dim//2].float()/dim)

b = torch.arange(0,9).reshape(1,-1)[...,::3]
a = torch.arange(0,9).reshape(1,-1)[...,1::3]
c = torch.arange(0,9).reshape(1,-1)[...,2::3]
a,b,c,torch.stack([b,a,c], dim=-1).flatten(-2)

In [ ]:
from modules import SwiGLU
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
x=torch.randn((128))*10
x.repeat_interleave(1,dim=0).shape
T=128
attn = torch.randn(4,4,128,128)
attn_mask = torch.tril(torch.ones(T,T,device=x.device,dtype=torch.bool))
attn = attn.masked_fill(~attn_mask,float("-inf"))
attn = F.softmax(attn,dim=-1)
attn.shape,attn.argmax(dim=-1).shape


In [ ]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            # nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim),
            # nn.BatchNorm1d(dim),
            nn.ReLU(),
            # nn.BatchNorm1d(dim),
            nn.Linear(dim,dim),
            # SwiGLU(hidden_dim,dim),
            # nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
m= MLP(dim=1,hidden_dim=1)

def fn1(x):
    return torch.sin(x) + torch.cos(x) + torch.log(x + 1) + torch.sqrt(x)+ +x +1 -torch.tanh(x) 

x=torch.arange(0,10,0.1)
label = fn1(torch.arange(0,10,0.1))
# plt.subplot(1,2,1)
plt.figure(figsize=(16, 6))
plt.plot(torch.arange(0,10,0.1),label.detach().numpy(),label='label')

optimizer = torch.optim.SGD(m.parameters(), lr=0.01)

epochs = 10000
intervl = 1000
for _ in range(epochs):
    y_pred = m(x.unsqueeze(-1))
    loss = torch.mean((fn1(x) - y_pred.squeeze(-1)) ** 2)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if _ % intervl == 0:
        print(f"Step {_}: loss = {loss.item()}")
        plt.plot(x,y_pred.squeeze(-1).detach().numpy(),label = f'y_pred{_}')
plt.legend()
plt.show()

for l in m.net:
    if isinstance(l, nn.Linear):
        print(l.weight)

In [ ]:
class MLP2(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            # nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, dim),
            # nn.BatchNorm1d(dim),
            nn.ReLU(),
            # nn.BatchNorm1d(dim),
            nn.Linear(dim,hidden_dim),
            nn.ReLU(),
            # nn.BatchNorm1d(dim),
            nn.Linear(hidden_dim,dim),
            
            # SwiGLU(hidden_dim,dim),
            # nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
m= MLP2(dim=1,hidden_dim=1)

x=torch.arange(0,10,0.1)
label = fn1(torch.arange(0,10,0.1))
# plt.subplot(1,2,1)
plt.figure(figsize=(16, 6))
plt.plot(torch.arange(0,10,0.1),label.detach().numpy(),label='label')

optimizer = torch.optim.SGD(m.parameters(), lr=0.01)

epochs = 10000
intervl = 1000
for _ in range(epochs):
    y_pred = m(x.unsqueeze(-1))
    loss = torch.mean((fn1(x) - y_pred.squeeze(-1)) ** 2)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if _ % intervl == 0:
        print(f"Step {_}: loss = {loss.item()}")
        plt.plot(x,y_pred.squeeze(-1).detach().numpy(),label = f'y_pred{_}')
plt.legend()
plt.show()

for l in m.net:
    if isinstance(l, nn.Linear):
        print(l.weight)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
class MLP3(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            # nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            # nn.BatchNorm1d(dim),
            # nn.ReLU(),
            # nn.BatchNorm1d(dim),
            # nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            # nn.BatchNorm1d(dim),
            nn.Linear(hidden_dim,hidden_dim),   
            nn.ReLU(),
            # nn.BatchNorm1d(dim),
            nn.Linear(hidden_dim,dim)
            # SwiGLU(hidden_dim,dim),
            # nn.Linear(hidden_dim, dim)
        )
    def forward(self, x):
        return self.net(x)
device = 'cuda'
def fn1(x):
    return torch.sin(x) + torch.cos(x) + torch.log(x + 1) + torch.sqrt(x)+ +x +1 -torch.tanh(x) 
m= MLP3(dim=1,hidden_dim=1000).to(device)

x=torch.arange(0,100,0.1).reshape(-1, 1).to(device) # 数据量增加十倍（之前是10，增加十倍后是100）(1000)
label = fn1(x)
# plt.subplot(1,2,1)
plt.figure(figsize=(16, 6))
plt.plot(x.cpu(),label.cpu().detach().numpy(),label='label')

epochs = 1000
lr = 0.00008 #之前是0.01   
intervl = 100
optimizer = torch.optim.SGD(m.parameters(), lr=lr) # 之前是sgd
for _ in range(epochs):
    y_pred = m(x)
    loss = torch.mean((fn1(x) - y_pred.squeeze(-1)) ** 2)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if _ % intervl == 0:
        print(f"Step {_}: loss = {loss.item()}")
        plt.plot(x.cpu(),y_pred.cpu().squeeze(-1).detach().numpy(),label = f'y_pred{_}')
plt.legend()
plt.show()
plt.figure(figsize=(16, 6))
for l in m.net:
    if isinstance(l, nn.Linear):
        print(l.weight.data, l.bias.data)    
        plt.hist(l.cpu().weight.data.numpy().flatten(), bins=30, alpha=0.5, label=f'Layer {l}')
        l.to(device)
plt.legend()
plt.show()
plt.figure(figsize=(16, 6))
for l in m.net:
    x = l(x.to(device).unsqueeze(0))
    plt.hist(x.data.cpu().numpy().flatten(), bins=30, alpha=0.5, label=f'Layer {l}')
plt.legend()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
class MLP4(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),nn.ReLU(),
            nn.Linear(hidden_dim,dim)
            )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42) # 非常容易卡到loss = 1021.1193237304688 没有batch时
device = 'cuda'
epochs = 500
lr = 0.01 #之前是0.01
#  在 Step 1000: loss = 1021.1193237304688

def fn1(x):
    return torch.sin(x) + torch.cos(x) + torch.log(x + 1) + torch.sqrt(x)+ +x +1 -torch.tanh(x) 
intervl = 100

m= MLP4(dim=1,hidden_dim=1000).to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=lr) # 

x = torch.arange(0,100,0.1).reshape(-1, 1).to(device) # 数据量增加十倍（之前是10，增加十倍后是100）(1000,1)
label = fn1(x) # (1000,1)
#归一化
scala = 1.0/x.shape[0]
x = x*scala
label = label*scala

plt.figure(figsize=(16, 6))
plt.ylim(0, 150)
plt.plot(x.cpu()/scala, label.cpu().detach().numpy()/scala, label='label')


for _ in range(epochs):

    y_pred = m(x)
    loss = torch.mean((label - y_pred) ** 2)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if _ % intervl == 0:
        print(f"Step {_}: loss = {loss.item()}")
        plt.plot(x.cpu()/scala, m(x).cpu().squeeze(-1).detach().numpy()/scala, label=f'y_pred{_}')
plt.legend()
plt.show()
plt.figure(figsize=(16, 6))
for l in m.net:
    if isinstance(l, nn.Linear):
        print(l.weight.data, l.bias.data)    
        plt.hist(l.cpu().weight.data.numpy().flatten()/scala, bins=30, alpha=0.5, label=f'Layer {l}')
        l.to(device)
plt.legend()
plt.show()
plt.figure(figsize=(16, 6))
for l in m.net:
    x = l(x.to(device))
    plt.hist(x.data.cpu().numpy().flatten()/scala, bins=30, alpha=0.5, label=f'Layer {l}')
plt.legend()
plt.show()
#求导
plt.figure(figsize=(16, 6))
# plt.scatter(x.detach().cpu().numpy()/scala, fn2(x).cpu().detach().numpy()/scala, label='derivative', alpha=0.5)
plt.legend()
plt.show()
